# Pipeline di validazione — Diagnosi NIA-AA a tre assi

Notebook di validazione locale: esegue la pipeline di calcolo diagnosi sul dataset reale "di punta" (FreeSurfer 4.3, CSF Elecsys, PET FBP) e ispeziona i risultati. **Non salva/carica nulla sul Datalake** — è solo un controllo di sanità prima di un uso più esteso.

## Le tre diagnosi NIA-AA calcolate

Il package `rules` calcola tre diagnosi indipendenti a partire dagli stessi dati di visita:

1. **DX1 — Diagnosi clinica** (`dx1_nia_clinical.py`, NIA-AA 2011 / McKhann-Albert-Sperling): CN / MCI / Dementia da CDR, MMSE, test di memoria, FAQ. Non guarda mai i biomarcatori.
2. **DX2 — Diagnosi biologica ATN** (`dx2_nia_atn.py`, NIA-AA 2018 / Jack et al.): stato Amiloide (A) / Tau (T) / Neurodegenerazione (N) da CSF, PET, volumi FreeSurfer. Non guarda mai la diagnosi clinica.
3. **DX3 — Staging combinato** (`dx3_nia_combined.py`, NIA-AA 2024 / Jack et al.): combina DX1 + DX2 (+ severità CDR-SB) in uno stadio 1-6 (dal preclinico all'AD conclamato), senza mai sovrascrivere gli assi 1 e 2.

Le tre diagnosi sono indipendenti per design: DX1 e DX2 non si condizionano a vicenda; DX3 le combina solo *dopo* che sono già state calcolate entrambe.

Ordine di esecuzione: `assign_dx1_batch` → `assign_dx2_batch` → `assign_dx3_batch` (i primi due sono in realtà indipendenti tra loro — solo il terzo richiede che i precedenti siano già stati calcolati).

In [1]:
import sys, os

# Il notebook vive dentro rules/, ma il package `rules` va importato come se
# ci si trovasse in DX_calculators/ (vedi README.md, sezione "Uso").
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pandas as pd
from dl_client import DatalakeClient
from rules import assign_dx1_batch, assign_dx2_batch, assign_dx3_batch
from rules.config import FLAG_JOIN_SEP

client = DatalakeClient()

In [2]:
# Dataset "di punta": FreeSurfer 4.3, CSF Elecsys, PET FBP — combinazione dichiarata
# "current focus dataset" in config.py (CSF_CUTOFFS["ELECSYS"], AMYLOID_PET_CUTOFFS["FBP"]).
# Elenco file disponibili verificato in prove.ipynb (cella di query_files).
OBJECT_NAME = "cleaned/merged/combination/subMERGE_4-3_elecsys_FBP_fxd_0.csv"
BUCKET = "aind"

df = client.download_file(object_name=OBJECT_NAME, bucket=BUCKET)
print(df.shape)
df.head()

(8222, 57)


,RID,COHORT,VISCODE,VISIT_MONTH,EXAMDATE,FLDSTRENG,FSVERSION,IMAGEUID,update_stamp,Ventricles%ICV,...,GENDER/male,MARRY/divorced,MARRY/married,MARRY/single,MARRY/widowed,RACE/Asian,RACE/Black,RACE/Mixed,RACE/Native_american,RACE/White
0,2.0,ADNI1,bl,0.0,2005-09-08,1.5T,4.3,35475.0,2023-07-07 04:59:40.0,5.957343,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,3.0,ADNI1,bl,0.0,2005-09-12,1.5T,4.3,32237.0,2023-07-07 04:59:40.0,4.404615,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3.0,ADNI1,m06,6.0,2006-03-13,1.5T,4.3,31863.0,2023-07-07 04:59:40.0,4.646381,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3.0,ADNI1,m12,12.0,2006-09-12,1.5T,4.3,35576.0,2023-07-07 04:59:40.0,4.732538,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,3.0,ADNI1,m24,24.0,2007-09-12,1.5T,4.3,88252.0,2023-07-07 04:59:40.0,5.118156,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
meta = client.get_metadata(object_name=OBJECT_NAME, bucket=BUCKET)
metadata_custom = meta["metadata"]["custom"]

# Attesi: elecsys / FBP / 4.3 — conferma che il file caricato è davvero quello giusto.
print("CSF_filter:", metadata_custom.get("CSF_filter"))
print("PET_filter:", metadata_custom.get("PET_filter"))
print("VOLUMES_filter:", metadata_custom.get("VOLUMES_filter"))

CSF_filter: elecsys
PET_filter: FBP
VOLUMES_filter: 4.3


In [11]:
df.columns

Index(['RID', 'COHORT', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'FLDSTRENG',
       'FSVERSION', 'IMAGEUID', 'update_stamp', 'Ventricles%ICV',
       'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV',
       'STATUS', 'ADAS11', 'ADAS13', 'CDRGLOB', 'CDRSB', 'FAQ', 'MMSE', 'MOCA',
       'RAVLT_immediate', 'AB40_CSF', 'AB4240_CSF', 'AB42_CSF', 'METHOD_CSF',
       'PT181_AB42_CSF', 'PT181_CSF', 'TTAU_AB42_CSF', 'TTAU_CSF',
       'ENTORHINAL_SUVR', 'INFERIOR_TEMPORAL_SUVR', 'METHOD_PET',
       'SUMMARY_SUVR', 'TAU_METAROI', 'TRACER', 'AGE', 'APOE', 'APOE_4',
       'DX/CN', 'DX/Dementia', 'DX/MCI', 'EDUCAT', 'ETHNICITY/latino',
       'ETHNICITY/not_latino', 'GENDER/female', 'GENDER/male',
       'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed',
       'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american',
       'RACE/White', 'DX1_clinical', 'DX1_clinical_detailed', 'DX1_flags',
       'DX1_protocol_used', 'DX1_memory_test_used', 'DX2_A', 'D

## Nota: protocollo ADNI (`ORIGPROT`) e indipendenza dal dataset

`DX1_clinical` viene calcolato risolvendo `ORIGPROT` (la fase di arruolamento ADNI del soggetto) riga per riga, per scegliere quale tabella storica di cutoff clinici usare (`ADNI_LM_CUTOFFS`, `ADNI_MMSE_GATES` in `config.py`). **`ORIGPROT` non sopravvive nella pipeline di merge finale** che produce questo dataset — non è un dato mancante per errore, semplicemente non è mai stato propagato oltre il download grezzo ADNIMERGE. Di conseguenza il protocollo cade sempre sul default `config.SYNTHETIC_DEFAULTS["protocol"] = "ADNI3"` ("modalità riferimento singolo").

La cella subito dopo il calcolo di DX1 verifica esplicitamente questo comportamento su `DX1_protocol_used`, invece di darlo per scontato.

**Nota per il futuro (non affrontata in questo notebook):** l'obiettivo di lungo periodo sarebbe rendere DX1 indipendente dal dataset analizzato — un set di cutoff standard come comportamento di default, con i criteri storici ADNI-phase-specifici come opzione avanzata esplicita anziché fallback silenzioso. È una riprogettazione a sé, che tocca il comportamento di default dell'intero modulo clinico — da discutere separatamente.

In [4]:
df = assign_dx1_batch(df)

# Verifica esplicita del comportamento "riferimento singolo" descritto sopra:
# ci aspettiamo di vedere solo "ADNI3", dato che ORIGPROT è assente da questo dataset.
df["DX1_protocol_used"].value_counts(dropna=False)

DX1_protocol_used
ADNI3    8222
Name: count, dtype: int64

In [5]:
df = assign_dx2_batch(df, metadata=metadata_custom)
df[["DX2_A", "DX2_T", "DX2_N"]].isna().sum()

DX2_A    3189
DX2_T    3305
DX2_N    2269
dtype: int64

In [6]:
df = assign_dx3_batch(df)
df["DX3_stage"].value_counts(dropna=False)

DX3_stage
NaN    7574
4.0     220
5.0     211
3.0     117
1.0      45
6.0      31
2.0      24
Name: count, dtype: int64

In [7]:
df["DX1_clinical"].value_counts(dropna=False)

DX1_clinical
Unknown     6308
Dementia     865
MCI          772
CN           277
Name: count, dtype: int64

In [8]:
df["DX2_ATN_label"].value_counts(dropna=False)

DX2_ATN_label
A?        3189
A-        2441
A+T-N-     659
A+T+N+     589
A+T-       570
A+T-N+     361
A+T+N-     209
A+T+       132
A+T?        52
A+T?N-      14
A+T?N+       6
Name: count, dtype: int64

In [9]:
df[["DX3_stage", "DX3_label"]].value_counts(dropna=False)

DX3_stage  DX3_label           
NaN        NaN                     7574
4.0        Mild AD dementia         220
5.0        Moderate AD dementia     211
3.0        Prodromal AD             117
1.0        Preclinical AD            45
6.0        Severe AD dementia        31
2.0        Preclinical AD + tau      24
Name: count, dtype: int64

In [10]:
def flag_counts(series: pd.Series) -> pd.Series:
    """Frequenza dei singoli flag (non delle combinazioni) in una colonna *_flags."""
    return series.dropna().str.split(FLAG_JOIN_SEP).explode().value_counts()

flag_counts(df["DX1_flags"])

DX1_flags
MISSING_LDELTOTAL                  8222
MEMORY_FALLBACK_RAVLT              7212
MISSING_CDGLOBAL                   6308
MISSING_CDMEMORY                   1914
MISSING_MEMORY_TEST                1010
ASSUMED_EDUC_BAND                   480
TIE_CDR_HIGH_NO_FUNC_IMPAIRMENT     245
TIE_CDR0_MEM_IMPAIRED               117
TIE_FAQ_MISSING                       8
MISSING_MMSE_GATE_SKIPPED             5
Name: count, dtype: int64

In [12]:
# Atteso: FS_VERSION_CUTOFF_PROXY_FROM_5.1 su tutte le righe con volume disponibile
# (dataset FS4.3, proxy dei cutoff 5.1 — vedi config.NEUROIMAGING_CUTOFFS["pct_icv"]["4.3"]).
# FS_VERSION_CUTOFFS_UNCONFIRMED non dovrebbe comparire più per nessuna versione nota.
flag_counts(df["DX2_flags"])

DX2_flags
METHOD_FROM_METADATA                24666
PARTIAL_ATN                          4422
FS_VERSION_CUTOFF_PROXY_FROM_5.1     3532
NO_BIOMARKERS                         532
CONFLICTING_AMYLOID_PET_CSF           276
Name: count, dtype: int64

In [13]:
# Controllo di sanità: per costruzione di combine_stage(), stage 1-2 devono venire
# solo da DX1_clinical=CN, stage 3 solo da MCI, stage 4-6 solo da Dementia.
# Qualunque cella fuori da questo pattern segnala un bug da investigare.
pd.crosstab(df["DX1_clinical"], df["DX3_stage"], dropna=False)

DX3_stage,1.0,2.0,3.0,4.0,5.0,6.0,NaN
DX1_clinical,,,,,,,
CN,45,24,0,0,0,0,208
Dementia,0,0,0,220,211,31,403
MCI,0,0,117,0,0,0,655
Unknown,0,0,0,0,0,0,6308


In [15]:
display(df.head())

,RID,COHORT,VISCODE,VISIT_MONTH,EXAMDATE,FLDSTRENG,FSVERSION,IMAGEUID,update_stamp,Ventricles%ICV,...,DX2_A,DX2_T,DX2_N,DX2_ATN_label,DX2_flags,DX2_method_source,DX2_method_uniform,DX3_stage,DX3_label,DX3_flags
0,2.0,ADNI1,bl,0.0,2005-09-08,1.5T,4.3,35475.0,2023-07-07 04:59:40.0,5.957343,...,None,None,False,A?,METHOD_FROM_METADATA|METHOD_FROM_METADATA|METH...,"{'csf_assay': 'metadata', 'amyloid_pet_tracer'...",True,NaN,None,None
1,3.0,ADNI1,bl,0.0,2005-09-12,1.5T,4.3,32237.0,2023-07-07 04:59:40.0,4.404615,...,True,False,False,A+T-N-,METHOD_FROM_METADATA|METHOD_FROM_METADATA|METH...,"{'csf_assay': 'metadata', 'amyloid_pet_tracer'...",True,NaN,None,None
2,3.0,ADNI1,m06,6.0,2006-03-13,1.5T,4.3,31863.0,2023-07-07 04:59:40.0,4.646381,...,None,None,True,A?,METHOD_FROM_METADATA|METHOD_FROM_METADATA|METH...,"{'csf_assay': 'metadata', 'amyloid_pet_tracer'...",True,NaN,None,None
3,3.0,ADNI1,m12,12.0,2006-09-12,1.5T,4.3,35576.0,2023-07-07 04:59:40.0,4.732538,...,True,True,False,A+T+N-,METHOD_FROM_METADATA|METHOD_FROM_METADATA|METH...,"{'csf_assay': 'metadata', 'amyloid_pet_tracer'...",True,NaN,None,None
4,3.0,ADNI1,m24,24.0,2007-09-12,1.5T,4.3,88252.0,2023-07-07 04:59:40.0,5.118156,...,None,None,True,A?,METHOD_FROM_METADATA|METHOD_FROM_METADATA|METH...,"{'csf_assay': 'metadata', 'amyloid_pet_tracer'...",True,NaN,None,None


## Riepilogo

- Nessun salvataggio o upload al Datalake eseguito in questo notebook.
- Annotare qui eventuali anomalie osservate nelle distribuzioni o nel cross-tab sopra.
- Follow-up rimandati (non affrontati qui):
  1. Indipendenza di DX1 dal dataset analizzato (default a un set di cutoff standard, protocollo ADNI-phase-specifico come opzione avanzata anziché fallback silenzioso) — vedi nota nella sezione "protocollo ADNI" sopra.
  2. Derivazione empirica di cutoff volumetrici FS-version-specifici — oggi tutte le versioni non-5.1 usano un proxy dei cutoff 5.1 (flag `FS_VERSION_CUTOFF_PROXY_FROM_5.1`), non cutoff validati indipendentemente.